# Melampo: misura dell'interprete delle immagini (VoxTell + TotalSegmentator)

Solo dati pubblici: Medical Segmentation Decathlon, Task06 Lung (tumori polmonari in TC, licenza CC BY-SA 4.0).

Prima di eseguire, nelle impostazioni del notebook (pannello a destra):
- **Accelerator**: GPU T4 x2 (oppure P100);
- **Internet**: on (serve la verifica del telefono sull'account Kaggle);
- se il repository è privato: **Add-ons → Secrets → GITHUB_TOKEN** con un token GitHub *fine-grained*, solo lettura, solo su `Melampo-AI`.

Poi *Run all*. Alla fine si scarica `/kaggle/working/probe.json` e il log, da portare a Claude.

In [ ]:
# 0. Internet: senza, pip e il download del dataset falliscono ("Temporary failure in name resolution")
import socket
try:
    socket.gethostbyname("pypi.org")
    print("Internet OK")
except OSError:
    raise SystemExit("Internet spento: pannello a destra -> Session options -> Internet: On "
                     "(richiede il telefono verificato in Settings), poi Run All di nuovo.")

In [ ]:
!pip install -q voxtell TotalSegmentator nibabel
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# VoxTell segnala un problema di memoria con PyTorch 2.9.x: in quel caso si scende alla 2.8
import torch, subprocess, sys
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.__version__.startswith("2.9"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.8.0", "torchvision==0.23.0"], check=True)
    print("installata torch 2.8.0: Run -> Restart & Run All")

In [ ]:
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repository pubblico: nessun token
url = f"https://{token}@github.com/francescoluciolattari/Melampo-AI.git" if token else "https://github.com/francescoluciolattari/Melampo-AI.git"
subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/melampo"], check=True)
print(open("/kaggle/working/melampo/scripts/imaging_probe.py").read()[:300])

In [ ]:
# Medical Segmentation Decathlon (CC BY-SA 4.0) sul registro AWS Open Data: bucket pubblico, file anche non
# compressi, quindi si scaricano solo i casi che servono (non gli 8,5 GB dell'archivio intero).
import os, glob, requests, xml.etree.ElementTree as ET
N = 5
DATA = "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
BUCKETS = ["https://msd-for-monai.s3.us-west-2.amazonaws.com", "https://msd-for-monai-eu.s3.eu-west-2.amazonaws.com"]
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

def listing(base, prefix="", delimiter=""):
    keys, prefixes, token = [], [], None
    while True:
        params = {"list-type": "2", "prefix": prefix}
        if delimiter: params["delimiter"] = delimiter
        if token: params["continuation-token"] = token
        r = requests.get(base, params=params, timeout=60)
        r.raise_for_status()
        root = ET.fromstring(r.content)
        keys += [k.text for k in root.findall("s3:Contents/s3:Key", NS)]
        prefixes += [p.text for p in root.findall("s3:CommonPrefixes/s3:Prefix", NS)]
        if root.findtext("s3:IsTruncated", namespaces=NS) != "true":
            return keys, prefixes
        token = root.findtext("s3:NextContinuationToken", namespaces=NS)

# 1) dati già aggiunti al notebook con "Add Input"?
images = sorted(glob.glob("/kaggle/input/**/imagesTr/lung_*.nii*", recursive=True))
if not images:
    for base in BUCKETS:
        try:
            keys, _ = listing(base, "Task06_Lung/")
        except Exception as e:
            print(base, "non raggiungibile:", e); continue
        imgs = sorted(k for k in keys if "/imagesTr/lung_" in k and k.endswith(".nii.gz") and "/._" not in k)
        if not imgs:
            _, top = listing(base, "", "/")
            print("nessun file Task06_Lung non compresso in", base, "- cartelle presenti:", top[:30])
            print("archivi:", [k for k in keys if k.endswith((".tar", ".tar.gz"))])
            continue
        for k in imgs[:N]:
            for kk in (k, k.replace("/imagesTr/", "/labelsTr/")):
                dest = os.path.join(DATA, kk)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                if not os.path.exists(dest):
                    with requests.get(f"{base}/{kk}", stream=True, timeout=600) as r:
                        r.raise_for_status()
                        with open(dest, "wb") as fh:
                            for chunk in r.iter_content(1 << 20): fh.write(chunk)
        images = sorted(glob.glob(f"{DATA}/**/imagesTr/lung_*.nii.gz", recursive=True))
        break
print(len(images), "immagini:", [os.path.basename(p) for p in images])
assert images, "nessun dato: mandare a Claude l'output di questa cella"

In [ ]:
import json, os
lobes = ["lung_upper_lobe_left", "lung_lower_lobe_left", "lung_upper_lobe_right", "lung_middle_lobe_right", "lung_lower_lobe_right"]
cases = [{"id": os.path.basename(p).split(".")[0], "image": p, "lesion": p.replace("imagesTr", "labelsTr"),
          "structure": lobes, "prompt": "tumore polmonare", "prompt_en": "lung tumor"} for p in images[:N]]
json.dump(cases, open("/kaggle/working/cases.json", "w"), indent=1)
print(len(cases), "casi")

In [ ]:
!cd /kaggle/working/melampo && python scripts/imaging_probe.py --cases /kaggle/working/cases.json --out /kaggle/working/probe.json --device cuda --fast-organs 2>&1 | tee /kaggle/working/probe.log

In [ ]:
import json
rows = json.load(open("/kaggle/working/probe.json"))
for r in rows:
    for k in ("prompt", "prompt_en"):
        if k in r:
            v = r[k]
            print(f"{r['id']:10} {k:9} {v['status']:10} Dice {v['dice']:.2f}  {v['seconds']:6.1f}s  "
                  f"misurato {v['measured_longest_axial_mm']} mm  riferimento {v['reference_longest_axial_mm']} mm")